In [1]:
import sys
import pathlib

import polars as pl

sys.path.append("../../")
from utils.io_utils import load_profiles, load_configs
from scipy.stats import ks_2samp
from statsmodels.stats.multitest import fdrcorrection
from utils.data_utils import split_meta_and_features

In [2]:
# load in raw data from
cfret_data_dir = pathlib.Path("../0.download-data/data/sc-profiles/cfret/").resolve(
    strict=True
)
cfret_profiles_path = (
    cfret_data_dir / "localhost230405150001_sc_feature_selected.parquet"
).resolve(strict=True)
cfret_feature_space_path = (
    cfret_data_dir / "cfret_feature_space_configs.json"
).resolve(strict=True)

# make results dir
results_dir = pathlib.Path("./results").resolve()
results_dir.mkdir(parents=True, exist_ok=True)

# signatures effect
signatures_results_dir = pathlib.Path(results_dir / "signatures")
signatures_results_dir.mkdir(exist_ok=True)

In [3]:
# setting parameters
treatment_col = "Metadata_cell_type_and_treatment"

# buscar parameters
healthy_label = "healthy_DMSO"
failing_label = "failing_DMSO"
on_off_signatures_method = "ks_test"

In [4]:
# loading profiles
cfret_df = load_profiles(cfret_profiles_path)

# load cfret_df feature space and update cfret_df
cfret_feature_space = load_configs(cfret_feature_space_path)
cfret_meta_features = cfret_feature_space["metadata-features"]
cfret_features = cfret_feature_space["morphology-features"]
cfret_df = cfret_df.select(pl.col(cfret_meta_features + cfret_features))

# add another metadata column that combins both Metadata_heart_number and Metadata_treatment
cfret_df = cfret_df.with_columns(
    (
        pl.col("Metadata_treatment").cast(pl.Utf8)
        + "_heart_"
        + pl.col("Metadata_heart_number").cast(pl.Utf8)
    ).alias("Metadata_treatment_and_heart")
)

# renaming Metadata_treatment to Metadata_cell_type + Metadata_treatment
cfret_df = cfret_df.with_columns(
    (
        pl.col("Metadata_cell_type").cast(pl.Utf8)
        + "_"
        + pl.col("Metadata_treatment").cast(pl.Utf8)
    ).alias(treatment_col)
)

# split features
cfret_meta, cfret_feats = split_meta_and_features(cfret_df)

# Display data
print(f"Dataframe shape: {cfret_df.shape}")
cfret_df.head()

Dataframe shape: (15793, 679)


Metadata_cell_id,Metadata_WellRow,Metadata_WellCol,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,Metadata_Image_Count_Cells,Metadata_ImageNumber,Metadata_Plate,Metadata_Well,Metadata_Cells_Number_Object_Number,Metadata_Cytoplasm_Parent_Cells,Metadata_Cytoplasm_Parent_Nuclei,Metadata_Nuclei_Number_Object_Number,Metadata_Site,Cytoplasm_AreaShape_BoundingBoxMinimum_X,Cytoplasm_AreaShape_Compactness,Cytoplasm_AreaShape_Eccentricity,Cytoplasm_AreaShape_Extent,Cytoplasm_AreaShape_FormFactor,Cytoplasm_AreaShape_MajorAxisLength,Cytoplasm_AreaShape_MeanRadius,Cytoplasm_AreaShape_MinorAxisLength,Cytoplasm_AreaShape_Perimeter,Cytoplasm_AreaShape_Solidity,Cytoplasm_AreaShape_Zernike_0_0,Cytoplasm_AreaShape_Zernike_1_1,Cytoplasm_AreaShape_Zernike_2_0,Cytoplasm_AreaShape_Zernike_2_2,Cytoplasm_AreaShape_Zernike_3_1,Cytoplasm_AreaShape_Zernike_4_0,Cytoplasm_AreaShape_Zernike_4_2,…,Nuclei_Texture_DifferenceVariance_Mitochondria_3_03_256,Nuclei_Texture_DifferenceVariance_PM_3_03_256,Nuclei_Texture_InfoMeas1_ER_3_00_256,Nuclei_Texture_InfoMeas1_ER_3_01_256,Nuclei_Texture_InfoMeas1_ER_3_02_256,Nuclei_Texture_InfoMeas1_ER_3_03_256,Nuclei_Texture_InfoMeas1_Hoechst_3_00_256,Nuclei_Texture_InfoMeas1_Hoechst_3_01_256,Nuclei_Texture_InfoMeas1_Hoechst_3_02_256,Nuclei_Texture_InfoMeas1_Hoechst_3_03_256,Nuclei_Texture_InfoMeas1_Mitochondria_3_00_256,Nuclei_Texture_InfoMeas1_Mitochondria_3_01_256,Nuclei_Texture_InfoMeas1_Mitochondria_3_02_256,Nuclei_Texture_InfoMeas1_Mitochondria_3_03_256,Nuclei_Texture_InfoMeas1_PM_3_00_256,Nuclei_Texture_InfoMeas1_PM_3_01_256,Nuclei_Texture_InfoMeas1_PM_3_02_256,Nuclei_Texture_InfoMeas1_PM_3_03_256,Nuclei_Texture_InfoMeas2_ER_3_01_256,Nuclei_Texture_InfoMeas2_ER_3_03_256,Nuclei_Texture_InfoMeas2_Hoechst_3_01_256,Nuclei_Texture_InfoMeas2_Hoechst_3_03_256,Nuclei_Texture_InfoMeas2_PM_3_01_256,Nuclei_Texture_InfoMeas2_PM_3_03_256,Nuclei_Texture_InverseDifferenceMoment_Actin_3_02_256,Nuclei_Texture_InverseDifferenceMoment_ER_3_01_256,Nuclei_Texture_InverseDifferenceMoment_ER_3_03_256,Nuclei_Texture_InverseDifferenceMoment_Mitochondria_3_03_256,Nuclei_Texture_InverseDifferenceMoment_PM_3_01_256,Nuclei_Texture_InverseDifferenceMoment_PM_3_03_256,Nuclei_Texture_SumEntropy_PM_3_01_256,Nuclei_Texture_SumVariance_ER_3_03_256,Nuclei_Texture_SumVariance_Hoechst_3_03_256,Nuclei_Texture_SumVariance_Mitochondria_3_01_256,Nuclei_Texture_SumVariance_PM_3_01_256,Metadata_treatment_and_heart,Metadata_cell_type_and_treatment
str,str,i64,i64,str,str,str,f64,f64,f64,f64,i64,i64,str,str,i64,i64,i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""210e4376efde9d557a5c60029bdda6…","""B""",2,9,"""failing""","""rejected""","""DMSO""",221.046761,137.115493,246.6028,109.285755,40,1,"""localhost230405150001""","""B02""",1,1,6,6,"""f00""",-1.35494,0.841229,0.648883,-0.850138,-1.045214,1.298358,0.376165,0.935101,1.530228,-0.983617,-0.261031,-0.299817,-0.721977,0.944725,0.161074,0.532329,1.845864,…,0.797095,0.359081,-0.173336,0.300041,0.217945,-0.039774,0.488531,0.472164,0.28659,0.464359,0.501649,0.507623,1.076663,0.741941,-0.696022,-0.178762,0.186741,0.158222,0.341595,0.50487,-0.440604,-0.426966,0.194372,-0.035117,0.400021,-0.619206,-0.393448,0.961214,0.406068,0.374039,-0.280532,-0.158967,-0.344804,-0.263653,-0.305486,"""DMSO_heart_9""","""failing_DMSO"""
"""cef18f209640ef8ae98ec110cfdcb6…","""B""",2,9,"""failing""","""rejected""","""DMSO""",690.596142,183.067828,716.170091,177.132195,40,1,"""localhost230405150001""","""B02""",2,2,7,7,"""f00""",0.657107,-0.850399,-0.584931,2.090925,1.263259,-0.021031,1.627957,0.944161,-0.085511,1.475345,2.164761,-0.688462,1.215015,1.499086,-0.770667,1.012721,0.6791,…,-1.

In [5]:
ref_df = cfret_df.filter(pl.col("Metadata_cell_type_and_treatment") == failing_label)
target_df = cfret_df.filter(pl.col("Metadata_cell_type_and_treatment") == healthy_label)

In [6]:
# apply ks test to each feature (single loop)
ks_stats = []
p_values = []
for feature in cfret_feats:
    stat, p_value = ks_2samp(ref_df[feature], target_df[feature])
    ks_stats.append(stat)
    p_values.append(p_value)

# FDR correction (vectorized)
_, p_values_fdr = fdrcorrection(p_values)

# Create DataFrame
ks_results_df = pl.DataFrame(
    {
        "feature": cfret_feats,
        "p_value": p_values,
        "ks_stat": ks_stats,
        "p_value_fdr_corrected": p_values_fdr,
    }
)

# Vectorized log transformation
ks_results_df = ks_results_df.with_columns(
    (-pl.col("p_value_fdr_corrected").log10()).alias("neg_log10_p_value")
)

# add a column for significance based on a threshold (e.g., 0.05)
# if lower than 0.05 then there should be a label known as "on" and higher than 0.05 should be "off"
ks_results_df = ks_results_df.with_columns(
    pl.when(pl.col("p_value_fdr_corrected") < 0.05)
    .then(pl.lit("on"))
    .otherwise(pl.lit("off"))
    .alias("signature")
)

# add a column of "channel" where we splitthe feature name and takethe first split
ks_results_df = ks_results_df.with_columns(
    pl.col("feature").str.split("_").list.get(0).alias("channel")
)

# save dataframe as csv
ks_results_df.write_csv(signatures_results_dir / "signature_importance.csv")

# display
print(ks_results_df.shape)
ks_results_df.head()

(657, 7)


feature,p_value,ks_stat,p_value_fdr_corrected,neg_log10_p_value,signature,channel
str,f64,f64,f64,f64,str,str
"""Cytoplasm_AreaShape_BoundingBo…",0.000128,0.063555,0.00015,3.825207,"""on""","""Cytoplasm"""
"""Cytoplasm_AreaShape_Compactnes…",0.784666,0.018836,0.789473,0.102663,"""off""","""Cytoplasm"""
"""Cytoplasm_AreaShape_Eccentrici…",3.8892e-47,0.211618,9.9039e-47,46.004195,"""on""","""Cytoplasm"""
"""Cytoplasm_AreaShape_Extent""",1.8732e-21,0.142288,3.1395e-21,20.503135,"""on""","""Cytoplasm"""
"""Cytoplasm_AreaShape_FormFactor""",0.784666,0.018836,0.789473,0.102663,"""off""","""Cytoplasm"""
